In [ ]:
# Cell 1: Imports & dataset download
import kagglehub, os, cv2, numpy as np
from skimage.feature import hog
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import joblib

path = kagglehub.dataset_download("fanconic/skin-cancer-malignant-vs-benign")
train_dir = os.path.join(path, "train")
test_dir  = os.path.join(path, "test")
img_size = (128, 128)

In [ ]:
# Cell 2: Load & preprocess images
def load_data(split_dir):
    X, y = [], []
    for label, cls in enumerate(['benign', 'malignant']):
        folder = os.path.join(split_dir, cls)
        for fname in os.listdir(folder):
            img = cv2.imread(os.path.join(folder, fname))
            img = cv2.resize(img, img_size)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            X.append(img); y.append(label)
    return np.array(X), np.array(y)

X_train_img, y_train = load_data(train_dir)
X_test_img, y_test   = load_data(test_dir)

In [ ]:
# Cell 3: HOG feature extraction
def extract_hog_features(images):
    return np.array([hog(img, orientations=9, pixels_per_cell=(8,8),
                          cells_per_block=(2,2), block_norm='L2-Hys') for img in images])

X_train = extract_hog_features(X_train_img)
X_test  = extract_hog_features(X_test_img)

In [ ]:
# Cell 4: Scale features & train SVM
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
model.fit(X_train, y_train)

In [ ]:
# Cell 5: Evaluate model
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
# Cell 6: Save trained model
joblib.dump(model, "skin_svm_model.pkl")
joblib.dump(scaler, "skin_scaler.pkl")

In [ ]:
# Cell 7: Gradio demo interface
import gradio as gr

def predict(img):
    if img.shape[-1] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    img = cv2.resize(img, img_size)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    feat = hog(img, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), block_norm='L2-Hys')
    feat = scaler.transform([feat])
    pred = model.predict(feat)[0]
    prob = model.predict_proba(feat)[0][pred]
    label = "Malignant" if pred == 1 else "Benign"
    return f"Prediction: {label}\nConfidence: {prob*100:.2f}%"

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(label="Upload Skin Lesion Image", type="numpy"),
    outputs=gr.Textbox(label="Prediction Output"),
    title="Skin Lesion Classifier (HOG + SVM)",
    description="Upload a skin lesion image to check if it shows signs of being benign or malignant."
)
demo.launch()